# **LightGBM**



LightGBM — это библиотека градиентного бустинга по деревьям решений, которая:

ускоряет обучение за счёт гистограмм и специальных трюков (GOSS, EFB);

строит деревья leaf-wise (по «наиболее выгодному листу»), а не уровень-за-уровнем, как XGBoost;

умеет нативно работать с категориальными признаками (их можно не one-hot-ить);

хорошо масштабируется на большие датасеты.

# Предобработка

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import pandas as pd
from data_preprocessing.DataForModel import build_target, split_data, preprocess_dataset

In [ ]:
name = input("Введите имя файла большими буквами: ")
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
df = preprocess_dataset(df)

In [ ]:
df = build_target(df, h=20, target_type="classification")

y = df["GoodTrade"]

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, target="GoodTrade", val_size=0.1, test_size=0.2, split_type="train_test")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# Обучение

Модель

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

lgbm_reg = LGBMRegressor(
    n_estimators=500,
    num_leaves=31,
    learning_rate=0.05,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    n_jobs=-1,
    random_state=42
)

lgbm_reg.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='l2',   # MSE на валидации
)

y_pred = lgbm_reg.predict(X_test)

print("R^2:", r2_score(y_test, y_pred))
print("Spearman corr:", spearmanr(y_test, y_pred))


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070100

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd

df_eval = df_signals.iloc[split_idx:].copy()  # та же выборка, что X_test / y_test
df_eval['GoodProb_true'] = y_test
df_eval['GoodProb_pred'] = y_pred

# 5 квантилей по предсказанию модели
df_eval['bucket'] = pd.qcut(df_eval['GoodProb_pred'], 5, labels=False)

print(
    df_eval.groupby('bucket')['ret_H'].mean()
)


bucket
0    0.001282
1    0.000706
2   -0.001457
3   -0.000672
4   -0.002434
Name: ret_H, dtype: float64
